In [ ]:
import os, json, pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
import shap
from thermoift import MLPostprocessing, plot_correlation_heatmap
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

In [ ]:
PLOT_FOLDER = "TabPFN_gamma_OUTPUTS"
target      = "gamma"

In [ ]:
with open(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")) as f:
    metrics = json.load(f)

features     = metrics["features"]
hpo_metric   = metrics["hpo_metric"]
cv_r2_mean   = metrics["cv_r2_mean"]
cv_rmse_mean = metrics["cv_rmse_mean"]

trials_df    = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_hpo_trials.csv"))
preds_df     = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"))
perm_df      = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_permutation_importance.csv"))
shap_rank_df = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_importance.csv"))
shap_values  = joblib.load(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_values.joblib"))
X_explain    = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_X_explain.csv"), index_col=0)

shap_arr = shap_values.values if hasattr(shap_values, "values") else np.asarray(shap_values)
if shap_arr.ndim == 3:
    shap_arr = shap_arr[:, :, 0]

print(f"Loaded all artifacts from {PLOT_FOLDER}/")
print(f"Features : {features}")
print(f"CV R²     : {cv_r2_mean:.6f}  |  CV RMSE: {cv_rmse_mean:.6f}")

In [ ]:
train_rows = preds_df[preds_df["split"] == "train"]
test_rows  = preds_df[preds_df["split"] == "test"]
val_rows   = preds_df[preds_df["split"] == "val"]

y_train      = pd.Series(train_rows["actual"].values, name=target)
y_train_pred = train_rows["predicted"].values
y_test       = pd.Series(test_rows["actual"].values, index=test_rows["idx"].values, name=target)
y_test_pred  = test_rows["predicted"].values
y_val        = pd.Series(val_rows["actual"].values, name=target)
y_val_pred   = val_rows["predicted"].values

post = MLPostprocessing(
    y_true=y_test,
    y_pred=y_test_pred,
    target=target,
    feature_names=features,
    datasets={
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)

In [ ]:
df = pd.read_csv("../../interfacial_results_dataset_A4.csv")
plot_correlation_heatmap(df, features, target, save_path=f"TabPFN_{target}_correlation", folder=PLOT_FOLDER)

In [ ]:
trials_plot   = trials_df.sort_values("trial_id").reset_index(drop=True)
LABEL_FS      = 18
TICK_FS       = 14
LEGEND_FS     = 16
SMALL_TICK_FS = 11

# 1) Convergence curve: trial loss + best-so-far overlay
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(trials_plot["trial_id"], trials_plot["loss"], color=ps.colors[1], alpha=0.55, linewidth=2.0, label="Trial loss")
ax.plot(trials_plot["trial_id"], trials_plot["loss"].cummin(), color=ps.colors[0], linewidth=3.0, linestyle="--", label="Best so far")
ax.set_xlabel("Trial", fontsize=LABEL_FS)
ax.set_ylabel(f"{hpo_metric.upper()} loss", fontsize=LABEL_FS)
ax.tick_params(axis="both", labelsize=TICK_FS)
ax.legend(fontsize=LEGEND_FS, frameon=True)
ps.apply_axis_style(ax)
fig.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_hpo_convergence", folder=PLOT_FOLDER)
plt.show()

# 2) Hyperparameter importance: |Pearson r| with loss
hp_cols  = [c for c in trials_plot.columns if c not in ("trial_id", "loss")]
valid_hp = [c for c in hp_cols if trials_plot[c].nunique() > 1]
hp_corr  = trials_plot[valid_hp].corrwith(trials_plot["loss"]).abs().sort_values()

fig_height = max(5, 0.55 * len(hp_corr))
fig, ax = plt.subplots(figsize=(9, fig_height), dpi=300)
ax.barh(hp_corr.index, hp_corr.values, color=ps.colors[0], edgecolor="white", linewidth=0.5)
ax.set_xlabel(r"$|\mathrm{Pearson}\ r|$ with loss", fontsize=LABEL_FS)
ax.set_ylabel("Hyperparameter", fontsize=LABEL_FS)
ax.tick_params(axis="x", labelsize=TICK_FS)
ax.tick_params(axis="y", labelsize=12)
ps.apply_axis_style(ax)
fig.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_hpo_hp_importance", folder=PLOT_FOLDER)
plt.show()

# 3) Parallel coordinates: top 8 HPs by variance, coloured by loss
top_hp  = trials_plot[valid_hp].std(numeric_only=True).sort_values(ascending=False).head(8).index.tolist()
norm_df = trials_plot[top_hp].copy()
for c in top_hp:
    mn, mx = norm_df[c].min(), norm_df[c].max()
    norm_df[c] = (norm_df[c] - mn) / (mx - mn + 1e-12)
loss_norm = (trials_plot["loss"] - trials_plot["loss"].min()) / (trials_plot["loss"].max() - trials_plot["loss"].min() + 1e-12)
cmap = plt.cm.RdYlGn_r
fig, ax = plt.subplots(figsize=(12, 5.5), dpi=300)
for i in range(len(norm_df)):
    ax.plot(range(len(top_hp)), norm_df.iloc[i][top_hp].values, color=cmap(loss_norm.iloc[i]), alpha=0.45, linewidth=1.2)
ax.set_xticks(range(len(top_hp)))
ax.set_xticklabels(top_hp, rotation=35, ha="right", fontsize=SMALL_TICK_FS)
ax.set_ylabel("Normalised value", fontsize=LABEL_FS)
ax.set_ylim(-0.05, 1.05)
ax.tick_params(axis="y", labelsize=TICK_FS)
sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=trials_plot["loss"].min(), vmax=trials_plot["loss"].max()))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(f"{hpo_metric.upper()} loss", fontsize=LABEL_FS)
cbar.ax.tick_params(labelsize=TICK_FS)
ps.apply_axis_style(ax)
fig.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_hpo_parallel_coords", folder=PLOT_FOLDER)
plt.show()

In [ ]:
post.plot_parity(model_name="TabPFN", save_path=f"TabPFN_{target}_parity_plot", folder=PLOT_FOLDER,
                 cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean)

In [ ]:
post.plot_residual_distribution(save_path=f"TabPFN_{target}_residual_distribution", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean)

In [ ]:
post.plot_residual_vs_predicted(save_path=f"TabPFN_{target}_residual_vs_predicted", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean)

In [ ]:
feature_labels = [ps.label_map.get(f, f) for f in features]

# 1) Permutation importance bar chart
fig, ax = ps.plot_init()
order        = perm_df.iloc[::-1]
order_labels = [ps.label_map.get(f, f) for f in order["feature"]]
ax.barh(order_labels, order["importance_mean"],
        xerr=order["importance_std"],
        color=ps.colors[0], edgecolor="white", linewidth=0.5)
ax.set_xlabel(r"Increase in RMSE when feature is permuted", fontsize=ps.label_fontsize)
ax.set_ylabel("Feature", fontsize=ps.label_fontsize)
ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_permutation_importance", folder=PLOT_FOLDER)
plt.show()

# 2) Mean |SHAP| bar plot
fig, ax = ps.plot_init()
shap.plots.bar(shap_values, show=False)
ax = plt.gca()
ax.set_xlabel(r"Mean $|\mathrm{SHAP}|$", fontsize=ps.label_fontsize)
ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_shap_bar", folder=PLOT_FOLDER)
plt.show()

# 3) Beeswarm summary plot
fig, ax = ps.plot_init()
shap.summary_plot(shap_values, X_explain, feature_names=feature_labels, show=False, plot_size=None)
ax = plt.gca()
ax.set_xlabel(r"SHAP value", fontsize=ps.label_fontsize)
ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_shap_summary", folder=PLOT_FOLDER)
plt.show()

# 4) Dependence plot for top feature
top_feature     = shap_rank_df.iloc[0]["feature"]
top_feature_idx = features.index(top_feature)
fig, ax = ps.plot_init()
shap.dependence_plot(
    top_feature_idx,
    shap_arr,
    X_explain,
    feature_names=feature_labels,
    ax=ax,
    show=False,
)
ax.set_xlabel(ps.label_map.get(top_feature, top_feature), fontsize=ps.label_fontsize)
ax.set_ylabel(rf"SHAP value for {ps.label_map.get(top_feature, top_feature)}", fontsize=ps.label_fontsize)
ps.apply_axis_style(ax)
plt.tight_layout()
ps.save_plot(fig, f"TabPFN_{target}_shap_dependence_{top_feature}", folder=PLOT_FOLDER)
plt.show()

In [ ]:
post.print_summary()